[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session4/BMEN35_Ex4_03_TinyStories_text_generation.ipynb)

# TinyStories: three short text generation experiments

**Run in Colab with the default CPU runtime.** Run the cells in order. The first model load downloads about 49 MB of weights. We use [TinyStories-1M](https://huggingface.co/roneneldan/TinyStories-1M), a small model trained to continue simple English stories.

A language model takes some starting text, called a **prompt**, and predicts scores for possible next tokens. After we choose a token, we add it to the end of the prompt and ask the model for another prediction. Repeating that process produces a **continuation**. TinyStories is a model for completing text, so our inputs are the beginnings of stories.

First we inspect one prediction. Then we compare two ways of choosing a token and look at every choice in a longer continuation. Finally we run two comparisons, changing either the temperature or the text of the prompt.

In [ ]:
# Install the library in this notebook's Python environment.
%pip -q install transformers

## 1. Load the tokenizer and the model

The **tokenizer** divides the text into pieces such as words, parts of words or punctuation. A piece may include a leading space. The tokenizer's vocabulary maps each piece to an integer **token ID**. Different models can use different vocabularies. We load the tokenizer that belongs to this model so both components interpret the IDs in the same way.

The **model** takes the token IDs and calculates a score for each possible next piece in its vocabulary. We keep the model and inputs on CPU. The `revision` points to a specific snapshot of the model on Hugging Face. Both calls use that snapshot so the tokenizer and the weights come from the same version.

In [ ]:
import torch                                               # PyTorch supplies tensors and runs the model.
from transformers import AutoModelForCausalLM, AutoTokenizer  # These classes load the matching model and tokenizer.

model_name = 'roneneldan/TinyStories-1M'                     # Use the small model intended to continue stories.
revision = '2ad005d9b1e727d4a7ed107816c44aff9f78935f'         # Fix the snapshot used for both downloads.

tokenizer = AutoTokenizer.from_pretrained(                  # Load vocabulary and tokenization rules.
    model_name, revision=revision,
)
model = AutoModelForCausalLM.from_pretrained(                # Load the network and its learned parameters.
    model_name, revision=revision, use_safetensors=True,    # Select the saved safetensors weights.
)
model.eval()                                               # Switch layers such as dropout to prediction behaviour.
print('Model device:', next(model.parameters()).device)    # Show the device used by the model weights.

## 2. Inspect the first prediction

Use the prompt **Once upon a time, Mia found a**. A likely next token might be an adjective or a noun. The model calculates a raw score for every vocabulary token. These raw scores are called **logits**. Softmax turns them into probabilities that sum to one across the full vocabulary.

The model's output has one row of logits at every input position. A row predicts the token **after** that position. The last row is the row we need when continuing the whole prompt. The output shape is `(number of prompts, number of input positions, vocabulary size)`. In `output.logits[0, -1]`, the index 0 chooses the only prompt and the index −1 chooses its last input position.

The cell uses `torch.topk` to find the five largest probabilities and the IDs they belong to. We convert those IDs back to readable text. The five probabilities are a small subset, so their total can be below 100%. Quotes let you see spaces at the start of token pieces.

**Before running.** Guess one possible next piece. After running, identify the highest probability candidate and read its probability. Could a lower probability candidate also finish the sentence in a sensible way?

In [ ]:
prompt = 'Once upon a time, Mia found a'           # Use a short beginning that the model can continue.
inputs = tokenizer(prompt, return_tensors='pt')    # Ask for PyTorch tensors, including the input IDs.

with torch.inference_mode():                      # Prediction does not need gradients for training.
    output = model(**inputs)                      # Pass the tokenizer's named tensors to the model.

next_token_scores = output.logits[0, -1]           # Select prompt 0, final input position, and keep all vocabulary scores.
probabilities = torch.softmax(next_token_scores, dim=0)  # Normalise over the single vocabulary dimension.
top_probabilities, top_ids = torch.topk(probabilities, k=5)  # Keep five candidate probabilities and matching IDs.

print('Prompt:', prompt)                           # Display the text that conditioned this prediction.
print('Input IDs:', inputs['input_ids'][0].tolist())  # Print the ordered token IDs for the only prompt.
print('Logit tensor shape:', tuple(output.logits.shape))  # Show how the three axes appear in this run.

for probability, token_id in zip(top_probabilities.tolist(), top_ids.tolist()):  # Read each probability with its ID.
    token = tokenizer.decode([token_id], clean_up_tokenization_spaces=False)     # Turn this one ID into a text piece.
    print(f'{token!r:15s} ID {token_id:5d} | {probability:.1%}')                   # Show the piece, its ID and its percentage.

## 3. Choose a token and repeat

**Decoding** is the procedure that turns the model's scores into a sequence of token choices. We use two choices here.

- **Greedy decoding** always takes a token with the highest probability at the current step. Suppose the probabilities for three candidates are 60%, 25% and 15%. Greedy decoding takes the 60% candidate. It makes that local choice and then calculates a new distribution for the updated text. It does not compare all complete stories before choosing.
- **Sampling** draws one token according to the probabilities. Think of 100 equally likely tickets, with 60 for the first token, 25 for the second and 15 for the third. The first token is more likely than either of the others. All three can be drawn. A **random seed** gives a repeatable starting point for the sequence of draws in the same setup.

The same model supplies the scores in both cases. If it chooses **little** after **Mia found a**, the next call predicts from **Mia found a little**. A different choice changes the context and can change everything that comes after it. The small example below runs both methods on the same three made-up probabilities before we use the real language model.

In [ ]:
candidates = ['little', 'red', 'small']                   # Give the made-up example three possible next tokens.
example_probabilities = torch.tensor([0.60, 0.25, 0.15])  # These probabilities add up to one.

best_index = example_probabilities.argmax().item()      # Find the index of the largest probability.
print('Greedy:', candidates[best_index])                 # Look up the token at that index.

torch.manual_seed(7)                                    # Start the random generator at a known position.
draws = torch.multinomial(example_probabilities, 6, replacement=True)  # Draw six times from this same distribution.
for draw in draws.tolist():                             # Convert the tensor of indices to a Python list.
    print('Sampled:', candidates[draw])                  # Show the token picked by each draw.

### Put the real generation call in a function

The six draws above repeat **the first choice** with the same probabilities. In real generation the distribution is updated after each selected token. The `generate` method runs that loop for us. It starts with the prompt IDs and returns them followed by the chosen IDs. If the prompt contains seven tokens, the new ones start at index 7. The slice in the function keeps only the new ones.

Use `do_sample=False` for greedy decoding. Use `do_sample=True` to sample. `max_new_tokens` sets an upper limit. If the model chooses its end token, the continuation can be shorter. The `show_steps` option keeps the scores for the longer trace below.

In [ ]:
def continue_story(prompt, do_sample, seed=7, temperature=1.0,
                   max_new_tokens=24, show_steps=False):
    # This function is used in the later experiments so the comparison settings stay consistent.
    inputs = tokenizer(prompt, return_tensors='pt')       # Convert the prompt to model inputs.
    prompt_length = inputs['input_ids'].shape[1]          # Count tokens along the sequence axis, which is axis 1.

    settings = {                                         # Collect the choices we will pass to generate.
        'max_new_tokens': max_new_tokens,                  # Limit the number of additional tokens.
        'do_sample': do_sample,                          # True samples; False chooses greedily with one beam.
        'num_beams': 1,                                   # Follow a single continuation.
        'pad_token_id': tokenizer.eos_token_id,           # Use the end token as padding for this model if needed.
    }

    if do_sample:                                        # These options control random decoding only.
        torch.manual_seed(seed)                          # Reset the random starting point for this call.
        settings['temperature'] = temperature            # Scale the next-token scores before sampling.
        settings['top_k'] = 20                            # Restrict the draw to the 20 highest scoring candidates.

    with torch.inference_mode():                         # We will only use the returned prediction, with no training.
        generated = model.generate(                      # Repeatedly choose a token and add it to the context.
            **inputs,                                    # Supply the named tensors from the tokenizer.
            **settings,                                  # Supply the decoding choices we collected above.
            return_dict_in_generate=True,                # Ask for a result with named fields such as sequences.
            output_scores=show_steps,                    # Keep per-step scores only when we want to display them.
        )

    continuation_ids = generated.sequences[0, prompt_length:]  # Skip the original prompt IDs.
    if show_steps:                                        # The longer example asks to see every token choice.
        print_token_steps(continuation_ids, generated.scores)  # Use the helper defined in the next subsection.

    return tokenizer.decode(continuation_ids, skip_special_tokens=True)  # Return readable new text.

In [ ]:
greedy_text = continue_story(prompt, do_sample=False)    # Choose a highest probability token at every step.
print('Prompt:', prompt)                                # Display the given text separately.
print('Greedy continuation:', repr(greedy_text))         # Display only what the model added.

### Follow all the decisions in a longer continuation

The short run above shows the selected text. A **trace** keeps the scores from each step as well. Every row of the trace below shows the **chosen token and its probability** and the **five highest probability candidates** for the same context. The five include the chosen token if it was among the five most likely. If a sampled token fell outside the top five, it still appears in the chosen column with its own probability.

The first row uses only the prompt as context. The second row uses the prompt plus the chosen token from row 1. An output such as `' cat'` may include a leading space. A displayed piece can also be part of a word, punctuation or an end token. The final displayed continuation leaves the end token out of the readable text.

For the **sampling** run, `generate` gives us scores after temperature and the top-20 filter. The table applies softmax to those scores. It shows the probabilities used in that run, which are different from the raw next-token probabilities we inspected in section 2. The five displayed probabilities need not add up to 100% because other tokens still have a chance.

**After running.** Find a step where the chosen token differs from the first candidate. Which token was actually added to the context for the next row? If the sample chose the first candidate every time, try another seed. Then set `do_sample=False` and run the same trace with greedy decoding.

In [ ]:
def print_token_steps(token_ids, step_scores):           # Display the actual choices and their alternatives.
    print('Step | Chosen token (probability) | Five highest probability candidates')

    for step_number, (chosen_id, scores) in enumerate(zip(token_ids, step_scores), start=1):  # Pair each choice with its scores.
        probabilities = torch.softmax(scores[0].float(), dim=0)  # Select the only prompt and normalise across token IDs.
        top_scores, top_ids = torch.topk(probabilities, k=5)       # Identify the five strongest candidates for this step.

        chosen_id = chosen_id.item()                            # Turn this single tensor value into a Python integer.
        chosen_piece = tokenizer.decode([chosen_id], clean_up_tokenization_spaces=False)  # Keep visible token spacing.
        chosen_probability = probabilities[chosen_id].item()   # Look up the probability of the token actually chosen.

        candidates = []                                        # Build a readable list for the last column.
        for token_id, probability in zip(top_ids.tolist(), top_scores.tolist()):  # Visit the candidates in ranked order.
            piece = tokenizer.decode([token_id], clean_up_tokenization_spaces=False)  # Decode one candidate's ID.
            candidates.append(f'{piece!r} ({probability:.1%})')   # Keep its token text beside its probability.

        candidate_text = ', '.join(candidates)                  # Combine the five entries into a single column.
        print(f'{step_number:>4} | {chosen_piece!r} ({chosen_probability:.1%}) | {candidate_text}')  # Print one generation step.

In [ ]:
traced_story = continue_story(             # Generate and keep the scores for each new token.
    prompt,                                # Use the same prompt as the first prediction.
    do_sample=True,                        # Draw each token from the filtered distribution.
    seed=7,                                # Choose the starting point for the random draws.
    temperature=0.8,                       # Put some extra weight on the higher scoring choices.
    max_new_tokens=24,                     # Show one trace row per token, up to 24.
    show_steps=True,                       # Enable the table of probabilities.
)

print('\nPrompt:', prompt)                # Print the initial context separately.
print('Continuation:', repr(traced_story)) # Show the text produced by the decisions above.

## 4. Compare two temperatures with the same seeds

**Temperature** is a positive number used when **sampling** the next token. At each step the model produces one raw score, or logit, per possible token. To apply temperature, divide the scores by that number **before** applying softmax. Softmax turns the scaled scores into sampling probabilities. Temperature changes the gaps between scores, which changes how likely the draw is to select each token.

- **Temperature 1** leaves the logits as they are.
- **Temperature below 1** makes score gaps larger. The higher scoring choices get more of the probability.
- **Temperature above 1** makes score gaps smaller. The probabilities become more even among tokens that survive the filter.

A positive temperature keeps the order of the logits. It changes the **probabilities** of sampling the candidates. Greedy decoding still picks the highest scoring candidate, so we do not set a temperature for greedy decoding in this notebook.

Try the two-logit calculation below. The first candidate stays ahead in all three rows. At 0.5 it is much more likely to be sampled than at 2.0. The story runs that follow also keep `top_k=20`, so at each step they draw only from the 20 highest scoring choices.

In [ ]:
example_logits = torch.tensor([2.0, 1.0])          # Give two possible tokens different raw scores.

for temperature in [0.5, 1.0, 2.0]:                # Choose one sharper, one unchanged and one flatter setting.
    scaled_logits = example_logits / temperature   # Change the gap between the scores before softmax.
    chances = torch.softmax(scaled_logits, dim=0)   # Convert the two scaled scores to probabilities.
    chance_a, chance_b = chances.tolist()           # Get the probabilities in the original candidate order.
    print(f'T={temperature:.1f} | first token {chance_a:.1%} | second token {chance_b:.1%}')

### Try the same idea on a story

Compare 0.5 and 1.1 with the same prompt, top-k and length limit. For each pair we reset the seed to the same value so the comparison starts from the same random state. The draws can still differ because the probabilities changed. Once a different token has been chosen, the model also sees different text for subsequent predictions. Repeating the comparison with three seeds gives three pairs to inspect.

**After running.** Choose a seed. Point to the first difference you can find. Did the object change, or did the stories diverge later? Use the exact output from that pair.

In [ ]:
for seed in [7, 13, 21]:                                        # Give the comparison three starting random states.
    low = continue_story(prompt, do_sample=True, seed=seed, temperature=0.5)   # Draw with more weight on top scores.
    high = continue_story(prompt, do_sample=True, seed=seed, temperature=1.1)  # Use the same seed with flatter scores.

    print(f'\nSeed {seed}')                                     # Keep the two runs with the same seed together.
    print('Temperature 0.5:', repr(low))                         # Show the continuation at the lower temperature.
    print('Temperature 1.1:', repr(high))                        # Show the output we can compare against it.

## 5. Change the text of the prompt

This time we keep the seed and temperature fixed. We add **red** to the prompt before the model chooses what Mia finds. Compare the prompt and continuation together. In the second run, **red** is already in the input and will appear only in the printed prompt unless the model also writes it later.

**Before running.** Guess an object that fits after **red**. **After running.** Find the first word or short phrase that completes each prompt. Record one concrete difference. If Mia finds the same thing in both rows, read farther to see where the stories diverge.

In [ ]:
for story_start in [prompt, 'Once upon a time, Mia found a red']:   # Compare the original prompt and its colour variant.
    continuation = continue_story(                                # Use the same decoding settings for both variants.
        story_start, do_sample=True, seed=7, temperature=0.8       # Change only the text in story_start.
    )
    print(f'\nPrompt: {story_start}')                             # Keep the exact input alongside its result.
    print('Continuation:', repr(continuation))                    # Print the generated text, with visible spaces.

## 6. Try your own comparison

- Change **red** to another colour and predict one plausible follow-up word. Run the prompt comparison again. Does the model pick an object you expected?
- Pick a new prompt. Keep that prompt fixed and repeat the temperature experiment for the same three seeds. Count how many different first objects you get at each temperature. If the text has no clear object, mark it as unclear. How would you make the comparison less sensitive to the three seeds chosen here?

[Hugging Face documentation on generation settings](https://huggingface.co/docs/transformers/en/main_classes/text_generation).